In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [ ]:
def longitude_flip(var):
    """ Convert longitude values from the -180:180 to 0:360 convention or vice versa.

        ** Only works for global data. Do not apply to data with a clipped longitude range **

        Parameters
        ----------
        var : Data Array
    """
    # get var info
    x,_=get_xy_coords(var) # extract original longitude values
    lon_name=x.name        # store name of longitude coordinate
    nx=len(x)              # longitude resolution

    # determine longitude format and create an array of new lons in opposite convention
    if min(x)<0:
        # if there are negative values, data is -180:180 and need to switch to 0:360
        new_lons=np.linspace((min(x)+180), (max(x)+180), nx)
    elif max(x)>180:
        # if the max value is >180, data is in 0:360 format and need to switch to -180:180
        new_lons=np.linspace((min(x)-180), (max(x)-180), nx)

    # shift the data by 180° of longitude
    nshift=nx//2
    var=var.roll({lon_name: nshift}, roll_coords=False)

    # update longitude coord with new values
    var=var.assign_coords({lon_name: new_lons})

    # add attributes documenting change
    timestamp=datetime.now().strftime("%B %d, %Y, %r")
    var.attrs['history']=f'flipped longitudes {timestamp}'
    var.attrs['original_lons']=x.values

    return(var)

def season_mean(ds, calendar='standard'):
    seas_mean = ds.groupby('time.season').mean(dim='time')
    return seas_mean

In [ ]:
### +++ LOAD DATA +++ ###

diags=['v_1000','sst','prec']
files={ 'obs':{}, 'ctrl':{}, '1sEnv.ca-only':{}, '1sEnv.ca-sa':{} }

for key in files.keys():    
    if key in ['obs']:
        files[key]['v_1000'] = 'obs.merra2.v1000.nc'
        files[key]['sst']    = 'obs.hadisst.sst.nc'
        files[key]['prec']   = 'obs.imerg.prec.nc'
        
    else:
        for varn in diags:
            files[key][varn] = f'aogcm.{key}.{varn}.nc'
            
# land mask
land_mask = xr.open_dataset('e2pt1.land_mask.nc').fgrnd

In [ ]:
## ORGANIZE DATA AND CALCULATE CLIMATOLOGIES

diags=['v_1000','sst','prec']
dat={ 'obs':{}, 'ctrl':{}, '1sEnv.ca-only':{}, '1sEnv.ca-sa':{} }
seas_mean={ 'obs':{}, 'ctrl':{}, '1sEnv.ca-only':{}, '1sEnv.ca-sa':{} }

# number of years to omit 
omit_yrs = 30
n_omit = omit_yrs * 12
tmax = 350 * 12
# lat/lon bounds
latmin = -55
latmax = 30
lonmin = 90
lonmax = 355

## Open and process timeseries files
print('Loading data for...')
for i,key in enumerate(dat.keys()):
    print(f'{key}')
    for varn in diags:
        
        # process obs files
        if key in ['obs']:
            if varn in ['v_1000']:
                ds = xr.open_dataset(files[key][varn], chunks={}).V[:,0,:,:].rename({'lev':'plm'})
                ds_flip = longitude_flip(ds)
                ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
                dat[key]['v_1000'] = ds_clip
                del ds
                del ds_flip
                del ds_clip
            if varn in ['prec']:
                ds = xr.open_dataset(files[key][varn], chunks={}).precipitation * 24 # convert from mm/hr to mm/day
                ds = ds.transpose('time','lat','lon')
                ds.attrs['units'] = 'mm/day'
                ds.attrs['Units'] = 'mm/day'
                ds_flip = longitude_flip(ds)
                ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
                dat[key]['prec'] = ds_clip
                del ds
                del ds_flip
                del ds_clip
            if varn in ['sst']:
                ds = xr.open_dataset(files[key][varn], chunks={}).sst.rename({'longitude':'lon', 'latitude':'lat'}).isel(lat=slice(None, None, -1))
                ds_flip = longitude_flip(ds)
                ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
                dat[key]['sst'] = ds_clip
                del ds
                del ds_flip
                del ds_clip
    
        # process model simulation files
        elif key in ['ctrl', '1sEnv.ca-only', '1sEnv.ca-sa']:
            ds = xr.open_dataset(files[key][varn], chunks={})[varn].isel(time=slice(n_omit,tmax))
            ds_flip = longitude_flip(ds)
            ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
            dat[key][varn] = ds_clip
            del ds
            del ds_flip
            del ds_clip
            
print('Done.')


## Calculate seasonal time-means
print('Calculating time means for...')
for key in seas_mean.keys():
    print(f'{key}')
    for varn in diags:
        seas_mean[key][varn] = season_mean(dat[key][varn])
print('Done.')


## Calculate zonal means of seasonal mean data
etp_zonal_mean={ 'obs':{}, 'ctrl':{}, '1sEnv.ca-only':{}, '1sEnv.ca-sa':{} }

# longitude bounds for Eastern Tropical Pacific
xmin1=-115
xmax1=-85

for key in seas_mean.keys():
    for varn in diags:
        etp_zonal_mean[key][varn] = seas_mean[key][varn].sel(lon=slice(xmin1,xmax1)).mean(dim=['lon'])
        if varn=='prec':
            etp_zonal_mean[key][varn] = etp_zonal_mean[key][varn].where(land_mask=>0.7, np.nan) # mask ocean data


In [ ]:
labels=['OBS','CTRL','CA-ONLY','CA+SA', 'OBS']
colors=['black', 'silver', '#ffb703', '#00b4d8']
line_kw={'ls':'-', 'lw':3} 
tkw={'axis': 'both', 'direction':'in', 'labelsize': 'large'} 
text_kw={'size': 'x-large', 'weight': 'bold',  'color': 'k', 'va':'center'}
legend_prop={'size':'x-large', 'weight':'bold'}
legend_kw={'labelcolor':'linecolor', 'ncols':1, 'frameon':False}

## ++ Make Plot ++ ##
fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(10,12), layout='constrained')

# SST
ax[0].text(-10, 29.4, '[115°W$-$85°W] Zonal Mean March-April-May Climate', rotation=0, ha='left', **text_kw)
ax[0].text(-12, 27.125, 'SST', rotation=90, ha='center', **text_kw)
for i,key in enumerate(etp_zonal_mean.keys()): 
    ax[0].plot(etp_zonal_mean[key]['sst'].lat, etp_zonal_mean[key]['sst'],
               c=colors[i], label=labels[i], **line_kw)
ax[0].set_ylabel('[°C]', weight='normal', size='x-large')
ax[0].set(ylim=[25.5,29.2])
ax[0].set_yticks([26,27,28,29])

# V-Wind
ax[1].text(-12, 1.25, '1000 hPa V Wind', rotation=90,  ha='center', **text_kw)
for i,key in enumerate(etp_zonal_mean.keys()): 
    ax[1].plot(etp_zonal_mean[key]['v_1000'].lat, etp_zonal_mean[key]['v_1000'],
               c=colors[i], label=labels[i], **line_kw)
ax[1].set_ylabel('[m/s]', weight='normal', size='x-large')
ax[1].set(ylim=[-2.5,5])
ax[1].set_yticks([-2,0,2,4])

# Precip
ax[2].text(-12, 5.5,'Precipitation', rotation=90,  ha='center', **text_kw)
for i,key in enumerate(etp_zonal_mean.keys()): 
    ax[2].plot(etp_zonal_mean[key]['prec'].lat, etp_zonal_mean[key]['prec'],
               c=colors[i], label=labels[i], **line_kw)
ax[2].legend(loc=1, prop=legend_prop, **legend_kw)
ax[2].set_ylabel('[mm/day]', weight='normal', size='x-large')
ax[2].set(ylim=[0,11])
ax[2].set_yticks([2,4,6,8,10])

for i,ax in enumerate(ax.flat):
    ax.axvline(x=0, color='black', linewidth=1, linestyle='--', label='_Hidden', zorder=0)
    ax.set(xlim=[-10, 15])
    ax.tick_params(**tkw)
    if i in [0,1]:
        ax.set_xticks([])
    if i==2:
        ax.set_xlabel('Latitude [°N]', weight='normal', size='large')